# 06 - Compare the two PD models

I compare the logistic scorecard and XGBoost on the same 2008 loans: ranking, PD error, a paired bootstrap, FICO and CLTV bands, drift, and TreeSHAP on held-out development loans.


In [1]:
# Load the saved comparison and 2008 predictions.
import os
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, brier_score_loss
from xgboost import DMatrix
from cohort import features_from

LOCAL_DATA = Path(r"E:\CRM Projects\Data")
if "POCKET_CRM_DATA" not in os.environ and LOCAL_DATA.is_dir():
    os.environ["POCKET_CRM_DATA"] = str(LOCAL_DATA)
DATA = Path(os.environ.get("POCKET_CRM_DATA", "./data")).expanduser().resolve()
OUT = DATA / "pocket1_model"
Y08 = "default_180_plus_2008"
comparisons = pd.read_csv(OUT / "xgb_development_comparison.csv")
oot_summary = pd.read_csv(OUT / "xgb_2008_comparison.csv")
cv = pd.read_csv(OUT / "xgb_cv_5fold.csv")
preds = pd.read_parquet(OUT / "xgb_2008_predictions.parquet")
raw = pd.read_parquet(OUT / "oot_2006q1_dec2007_2008_raw.parquet")
xgb_info = joblib.load(OUT / "xgb_challenger.joblib")
champion = joblib.load(OUT / "woe_logistic_champion.joblib")
known = preds.dropna(subset=[Y08]).copy()
known[Y08] = known[Y08].astype(int)
print("OOT loans:", len(preds), "| verified:", len(known),
      "| unresolved:", len(preds)-len(known))

OOT loans: 253343 | verified: 253094 | unresolved: 249


In [2]:
# Ranking next to PD error.
show = ["split", "model", "loans", "defaults", "AUC", "PR_AUC", "KS", "Gini",
        "Brier", "log_loss", "observed", "predicted"]
summary = pd.concat([comparisons, oot_summary], ignore_index=True)
print(summary[show].round(5).to_string(index=False))
print("\nTraining CV stability (XGBoost; fold held-out scores):")
print(cv[["AUC", "KS", "Brier", "log_loss"]].agg(["mean", "std"]).round(5))

same = oot_summary.set_index("model")
delta_auc = float(same.loc["XGBoost", "AUC"] - same.loc["Logistic", "AUC"])
delta_brier = float(same.loc["XGBoost", "Brier"] - same.loc["Logistic", "Brier"])
print(f"\nXGBoost minus logistic: OOT delta AUC={delta_auc:+.5f}, delta Brier={delta_brier:+.5f}")


     split    model  loans  defaults     AUC  PR_AUC      KS    Gini   Brier  log_loss  observed  predicted
validation  XGBoost  57238       113 0.83335 0.00944 0.54878 0.66669 0.00196   0.01273   0.00197    0.00199
validation Logistic  57238       113 0.82361 0.00775 0.52576 0.64722 0.00196   0.01290   0.00197    0.00201
      test  XGBoost  57238       112 0.79253 0.00997 0.43410 0.58506 0.00195   0.01295   0.00196    0.00198
      test Logistic  57238       112 0.77899 0.00749 0.42986 0.55798 0.00195   0.01311   0.00196    0.00200
  2008 OOT  XGBoost 253094      1975 0.76766 0.02250 0.40382 0.53532 0.00772   0.04461   0.00780    0.00325
  2008 OOT Logistic 253094      1975 0.76235 0.02105 0.39174 0.52469 0.00772   0.04447   0.00780    0.00328

Training CV stability (XGBoost; fold held-out scores):
          AUC       KS    Brier  log_loss
mean  0.80251  0.51821  0.00196   0.01309
std   0.01972  0.04429  0.00001   0.00024

XGBoost minus logistic: OOT delta AUC=+0.00531, delta Brier=-

In [3]:
# Paired bootstrap: resample the same loans for both models.
N_BOOT, SEED = 300, 42
rng = np.random.default_rng(SEED)
y = known[Y08].to_numpy()
p_l = known["logistic_PD"].to_numpy(dtype=float)
p_x = known["xgb_PD"].to_numpy(dtype=float)
pos, neg = np.flatnonzero(y == 1), np.flatnonzero(y == 0)
diffs = []
for _ in range(N_BOOT):
    idx = np.concatenate([rng.choice(pos, len(pos), replace=True),
                          rng.choice(neg, len(neg), replace=True)])
    diffs.append([roc_auc_score(y[idx], p_x[idx]) - roc_auc_score(y[idx], p_l[idx]),
                  brier_score_loss(y[idx], p_x[idx]) - brier_score_loss(y[idx], p_l[idx])])
bs = np.asarray(diffs)
uncertainty = pd.DataFrame({
    "metric": ["AUC difference (XGB-logistic)", "Brier difference (XGB-logistic)"],
    "observed_difference": [delta_auc, delta_brier],
    "CI_2.5pct": np.quantile(bs, .025, axis=0),
    "CI_97.5pct": np.quantile(bs, .975, axis=0)
})
print(uncertainty.round(6).to_string(index=False))


                         metric  observed_difference  CI_2.5pct  CI_97.5pct
  AUC difference (XGB-logistic)             0.005314   0.001817    0.009198
Brier difference (XGB-logistic)            -0.000006  -0.000010   -0.000002


In [4]:
# Deciles for each model, sorted by its own PD.
cal_tables = []
for model, pd_col in [("Logistic", "logistic_PD"), ("XGBoost", "xgb_PD")]:
    d = known[[Y08, pd_col]].copy()
    d["decile"] = pd.qcut(-d[pd_col], 10, labels=False, duplicates="drop") + 1
    tab = (d.groupby("decile").agg(loans=(Y08, "size"), defaults=(Y08, "sum"),
           observed_PD=(Y08, "mean"), predicted_PD=(pd_col, "mean"))
           .reset_index().assign(model=model))
    cal_tables.append(tab)
calibration = pd.concat(cal_tables, ignore_index=True)
calibration["lift"] = calibration["observed_PD"] / known[Y08].mean()
print(calibration[["model", "decile", "loans", "defaults", "observed_PD", "predicted_PD", "lift"]]
      .round(5).to_string(index=False))

# Positive gap means average PD is too high; negative means too low.
cal_level = (summary.loc[summary["split"].eq("2008 OOT"),
                         ["model", "observed", "predicted"]].copy())
cal_level["PD_gap"] = cal_level["predicted"] - cal_level["observed"]
print("\nOOT average PD gaps:\n", cal_level.round(5).to_string(index=False))

   model  decile  loans  defaults  observed_PD  predicted_PD    lift
Logistic       1  25332       611      0.02412       0.01240 3.09091
Logistic       2  25424       432      0.01699       0.00639 2.17748
Logistic       3  25178       301      0.01195       0.00443 1.53200
Logistic       4  25353       204      0.00805       0.00315 1.03114
Logistic       5  25403       175      0.00689       0.00228 0.88281
Logistic       6  25169       116      0.00461       0.00160 0.59062
Logistic       7  25310        90      0.00356       0.00107 0.45569
Logistic       8  25543        29      0.00114       0.00078 0.14549
Logistic       9  25161        10      0.00040       0.00048 0.05093
Logistic      10  25221         7      0.00028       0.00024 0.03557
 XGBoost       1  25310       630      0.02489       0.01476 3.18980
 XGBoost       2  25309       433      0.01711       0.00645 2.19244
 XGBoost       3  25309       282      0.01114       0.00403 1.42787
 XGBoost       4  25310       225 

In [5]:
# Check broad FICO and CLTV bands.
segment_cols = [f for f in ("fico", "cltv") if f in raw.columns]
seg = known.merge(raw[["loan_id"] + segment_cols], on="loan_id", how="left", validate="one_to_one")
segment_rows = []
for f, edges, labels in [("fico", [0, 620, 680, 740, np.inf], ["<620", "620-679", "680-739", "740+"]),
                         ("cltv", [0, 60, 80, 95, np.inf], ["<60", "60-79", "80-94", "95+"])]:
    if f not in segment_cols:
        continue
    values = pd.to_numeric(seg[f], errors="coerce")
    buckets = pd.cut(values, bins=edges, labels=labels, right=False)
    for group, d in seg.groupby(buckets, observed=True):
        if len(d) < 30:
            continue
        for name, pd_col in [("Logistic", "logistic_PD"), ("XGBoost", "xgb_PD")]:
            row = dict(segment=f, band=str(group), model=name, loans=len(d),
                       defaults=int(d[Y08].sum()), observed_PD=d[Y08].mean(),
                       predicted_PD=d[pd_col].mean(),
                       Brier=brier_score_loss(d[Y08], d[pd_col]))
            if d[Y08].nunique() == 2:
                row["AUC"] = roc_auc_score(d[Y08], d[pd_col])
            else:
                row["AUC"] = np.nan
            segment_rows.append(row)
segments = pd.DataFrame(segment_rows)
print(segments.round(5).to_string(index=False) if not segments.empty else "No sizable segments")

segment    band    model  loans  defaults  observed_PD  predicted_PD   Brier     AUC
   fico    <620 Logistic   9528       192      0.02015       0.00946 0.01983 0.57709
   fico    <620  XGBoost   9528       192      0.02015       0.01132 0.01975 0.60331
   fico 620-679 Logistic  49511       809      0.01634       0.00733 0.01611 0.64029
   fico 620-679  XGBoost  49511       809      0.01634       0.00795 0.01609 0.64896
   fico 680-739 Logistic  80811       681      0.00843       0.00324 0.00837 0.67110
   fico 680-739  XGBoost  80811       681      0.00843       0.00288 0.00837 0.67877
   fico    740+ Logistic 113022       292      0.00258       0.00102 0.00258 0.78430
   fico    740+  XGBoost 113022       292      0.00258       0.00077 0.00258 0.79835
   cltv     <60 Logistic  59962       131      0.00218       0.00105 0.00217 0.83349
   cltv     <60  XGBoost  59962       131      0.00218       0.00090 0.00217 0.84284
   cltv   60-79 Logistic  78957       620      0.00785       0.00

In [6]:
# Roll one-hot gain back up to the original field.
importance = xgb_info["model"].get_booster().get_score(importance_type="total_gain")
features = xgb_info["features"]
imp_rows = []
for feature in features:
    prefix = f"cat__{feature}_"
    score = (importance.get(f"num__{feature}", 0.0) +
             sum(v for name, v in importance.items() if name.startswith(prefix)))
    imp_rows.append({"feature": feature, "XGB_total_gain": score})
imp = pd.DataFrame(imp_rows).sort_values("XGB_total_gain", ascending=False)
print("XGBoost total gain:\n", imp.head(15).round(2).to_string(index=False))

coef = pd.DataFrame({"feature": champion["selected"],
                     "logistic_WoE_coefficient": champion["model"].coef_[0]})
coef["abs_coefficient"] = coef["logistic_WoE_coefficient"].abs()
print("\nLogistic WoE coefficients:\n",
      coef.sort_values("abs_coefficient", ascending=False).round(4).to_string(index=False))
print("\nEnforced XGBoost monotonic directions:", xgb_info["monotonic_constraints"])
print("Calibration used:", xgb_info["calibration_method"],
      "| Class weighting used:", xgb_info["use_class_weight"])

XGBoost total gain:
                 feature  XGB_total_gain
                   fico         2325.33
 original_interest_rate          874.94
                    ltv          700.25
           original_upb          595.02
                   cltv          581.59
                    dti          496.15
            seller_name          322.42
           loan_purpose          211.86
     original_loan_term          147.91
          property_type           58.32
                channel           51.58
       occupancy_status            2.13
interest_only_indicator            0.00

Logistic WoE coefficients:
                feature  logistic_WoE_coefficient  abs_coefficient
                  fico                   -0.8172           0.8172
                   ltv                   -0.6294           0.6294
original_interest_rate                   -0.5289           0.5289
                   dti                   -0.4430           0.4430
           seller_name                   -0.3730           0

In [7]:
# TreeSHAP on up to 300 held-out 2007 loans, before calibration.
test_loans = pd.read_parquet(DATA / "model_2005q1_dec2006_2007.parquet")
test_ids = pd.read_parquet(OUT / "loan_splits.parquet")
test_loans = test_loans.merge(test_ids, on="loan_id", validate="one_to_one")
test_loans = test_loans.loc[test_loans["split"].eq("test")].head(300)
x = features_from(test_loans, xgb_info["features"])
encoded = xgb_info["encoder"].transform(x)
model = xgb_info["model"]
booster = model.get_booster()
dm = DMatrix(encoded, feature_names=list(encoded.columns))
used_trees = (0, model.best_iteration + 1)  # Match the early-stopped model used for PD.
contrib = booster.predict(dm, pred_contribs=True, iteration_range=used_trees)
margin = booster.predict(dm, output_margin=True, iteration_range=used_trees)
if not np.allclose(contrib.sum(axis=1), margin, atol=1e-4):
    raise AssertionError("SHAP contributions do not sum to raw model output")
shap_values = pd.DataFrame(contrib[:, :-1], columns=encoded.columns)
shap_by_feature = pd.DataFrame({f: shap_values.loc[:, [c for c in shap_values if c == f"num__{f}" or
    c.startswith(f"cat__{f}_")]].sum(axis=1) for f in xgb_info["features"]})
shap_importance = shap_by_feature.abs().mean().sort_values(ascending=False)
print("Mean absolute TreeSHAP log-odds contribution:\n", shap_importance.head(10).round(4))
print("First example loan, top signed contributions:\n",
      shap_by_feature.iloc[0].reindex(shap_by_feature.iloc[0].abs().sort_values(ascending=False).index)
      .head(5).round(4))


Mean absolute TreeSHAP log-odds contribution:
 fico                      0.7580
ltv                       0.3044
dti                       0.2426
cltv                      0.2254
original_interest_rate    0.1873
loan_purpose              0.1651
original_upb              0.1455
seller_name               0.1249
original_loan_term        0.1045
channel                   0.0231
dtype: float32
First example loan, top signed contributions:
 fico            0.6893
loan_purpose    0.2439
cltv           -0.2293
dti            -0.2198
seller_name    -0.1425
Name: 0, dtype: float32


In [8]:
# Has the mix of scores moved since the development test split?
dev = pd.read_parquet(OUT / "xgb_development_predictions.parquet")
dev = dev.loc[dev["split"].eq("test")]
drift_rows = []
for model_name, column in [("Logistic", "logistic_PD"), ("XGBoost", "xgb_PD")]:
    base, later = dev[column].to_numpy(dtype=float), preds[column].to_numpy(dtype=float)
    edges = np.unique(np.r_[-np.inf, np.quantile(base, np.linspace(.1, .9, 9)), np.inf])
    a = np.histogram(base, bins=edges)[0] / len(base)
    b = np.histogram(later, bins=edges)[0] / len(later)
    # Empty band: use a small floor for the log ratio.
    a, b = np.maximum(a, 1e-6), np.maximum(b, 1e-6)
    drift_rows.append(dict(model=model_name, score_PSI=np.sum((b-a)*np.log(b/a))))
score_drift = pd.DataFrame(drift_rows)
print("Development test vs 2008 score PSI:\n", score_drift.round(4))


Development test vs 2008 score PSI:
       model  score_PSI
0  Logistic     0.2892
1   XGBoost     0.1274


In [9]:
uncertainty.to_csv(OUT / "final_model_paired_bootstrap.csv", index=False)
calibration.to_csv(OUT / "final_model_calibration_deciles.csv", index=False)
if not segments.empty:
    segments.to_csv(OUT / "final_model_segment_review.csv", index=False)
imp.to_csv(OUT / "final_model_xgb_importance.csv", index=False)
coef.to_csv(OUT / "final_model_logistic_coefficients.csv", index=False)
shap_importance.to_csv(OUT / "final_model_xgb_shap_importance.csv")
score_drift.to_csv(OUT / "final_model_score_psi.csv", index=False)
print("Saved Notebook 6 review outputs to", OUT)

Saved Notebook 6 review outputs to E:\CRM Projects\Data\pocket1_model


## My take

I would keep the logistic scorecard as the benchmark and track XGBoost as a challenger. On the 2008 loans, XGBoost's AUC was 0.768 versus 0.762, a gap of +0.005 (paired bootstrap 95% interval: +0.002 to +0.009). Its Brier score improved by about 0.000006 (95% interval: 0.000002 to 0.000010 lower), while log loss was slightly worse (0.04461 versus 0.04447). It found 630 defaults in its highest-risk decile versus 611 for logistic. These are small gains to weigh against the extra complexity. Both models scored the same loans, but the scorecard selected 7 fields and XGBoost used all 13, so this comparison cannot isolate an algorithm advantage from an input advantage.

XGBoost's score PSI was 0.127 versus 0.289 for logistic; that describes different shifts in their scores, not better performance. Neither model's 2008 probabilities are ready as final PDs: 0.780% of loans defaulted, versus average predictions of 0.325% for XGBoost and 0.328% for logistic. In XGBoost's highest-risk decile, 2.489% defaulted against 1.476% predicted. The 2008 housing downturn likely contributed to the gap, and these models contain no macroeconomic inputs. Original interest rate and seller mix also shifted (bin PSI 2.59 and 2.17). I would inspect those bins, including unseen seller categories, and recalibrate on separate labeled loans before selecting a final PD model.
